In [60]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM


In [61]:
#Loading feature vectors
vision_features = torch.load('../Encoder/features.pt')
labels = torch.load('../Encoder/labels.pt')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
vision_features = vision_features.to(device)

In [62]:
#Text tokenizer and Encoder
from transformers import AutoTokenizer, GPT2Tokenizer

# Use GPT2 tokenizer for consistency with LLM
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

# Use BiomedNLP for encoding text semantics
text_encoder = AutoModel.from_pretrained("microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract").to(device)

#Freeze text encoder parameters
for param in text_encoder.parameters():
    param.requires_grad = False


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 30997.38it/s]
BertModel LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [63]:
#LLM report generation

llm = AutoModelForCausalLM.from_pretrained("gpt2").to(device)

for p in llm.parameters():
    p.requires_grad = False

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 4675.36it/s]


In [64]:
#Diemensions
vision_dim = vision_features.shape[1]
text_dim = text_encoder.config.hidden_size
hidden_dim = 512
llm_dim = llm.config.n_embd

In [65]:
#Projection layer
image_projection = nn.Linear(vision_dim, hidden_dim).to(device)
text_projection = nn.Linear(text_dim, hidden_dim).to(device)
llm_projection = nn.Linear(hidden_dim, llm_dim).to(device)

#Fusion projection layer
fusion_projection = nn.Linear(hidden_dim * 2, hidden_dim).to(device)


In [66]:
#Simple context embedding (clinical report start)
clinical_context = nn.Parameter(torch.randn(1, hidden_dim)).to(device)


In [67]:
#Fusion module - improved with concatenation and projection
def fuse(vision_emb, text_emb):
    # Concatenate vision and text embeddings
    fused = torch.cat([vision_emb, text_emb], dim=-1)
    # Apply fusion projection layer
    return fusion_projection(fused)


In [ ]:
def encode_text(text_inputs):
    """
    Encode list of text strings into embeddings using PubMedBERT
    """
    inputs = tokenizer(
        text_inputs,
        padding=True,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = text_encoder(**inputs)

    # Use CLS token embedding (standard for BERT-style models)
    text_emb = outputs.last_hidden_state[:, 0, :]  # [B, text_dim]

    return text_emb

In [68]:
#Full forward pass with multimodal fusion

def forward(vision_feats, text_inputs):
    """
    Forward pass that fuses vision and text modalities for report generation
    
    Args:
        vision_feats: Vision features from encoder [B, vision_dim]
        text_inputs: List of text strings describing the context
    
    Returns:
        Fused multimodal embedding ready for LLM
    """
    
    #Text encoding
    text_emb = encode_text(text_inputs)
    text_emb = text_projection(text_emb)

    #Vision encoding
    vision_emb = image_projection(vision_feats)

    #Multimodal Fusion
    fused_emb = fuse(vision_emb, text_emb)

    #Project to LLM space
    llm_input = llm_projection(fused_emb)

    # Return fused embedding (used for generation)
    return llm_input


In [69]:
# =========================
# 6. CLINICAL PROMPT (REPORT STYLE)
# =========================
def build_prompt():
    return (
        "Clinical Impression: Liver assessment from abdominal CT imaging reveals "
    )


In [70]:
# =========================
# 11. IMPROVED TEXT GENERATION - SIMPLE VISION TO REPORT
# =========================
def generate_text(vision_feats, max_len=80):
    """
    Generate a medical report from vision features
    """
    
    # Project vision features
    vision_emb = image_projection(vision_feats)  # [B, hidden_dim]
    
    # Fuse with clinical context
    context_repeated = clinical_context.expand(vision_emb.shape[0], -1)  # [B, hidden_dim]
    fused = fuse(vision_emb, context_repeated)  # [B, hidden_dim]
    
    # Project to LLM input space
    llm_input_emb = llm_projection(fused)  # [B, llm_dim]
    
    if llm_input_emb.dim() == 1:
        llm_input_emb = llm_input_emb.unsqueeze(0)
    
    # Initialize generated sequence
    generated = llm_input_emb.unsqueeze(1)  # [B, 1, llm_dim]
    
    output_tokens = []
    
    # Generation loop
    for step in range(max_len):
        with torch.no_grad():
            try:
                # Get logits from LLM
                out = llm(inputs_embeds=generated, return_dict=True)
                next_token_logits = out.logits[:, -1, :]  # [B, vocab_size]
                
                # Get next token (greedy decoding)
                next_token = torch.argmax(next_token_logits, dim=-1)  # [B]
                token_id = next_token[0].item()
                
                output_tokens.append(token_id)
                
                # Stop if we generate end token
                if token_id == tokenizer.eos_token_id:
                    break
                
                # Get embedding for next token from GPT-2 and append
                next_emb = llm.transformer.wte(next_token).unsqueeze(1)  # [B, 1, llm_dim]
                generated = torch.cat([generated, next_emb], dim=1)
                
            except Exception as e:
                print(f"Error at step {step}: {e}")
                break
    
    # Decode the generated tokens
    if output_tokens:
        report = tokenizer.decode(output_tokens, skip_special_tokens=True)
    else:
        report = "[No report generated]"
    
    return report


In [71]:
# =========================
# 12. RUN - GENERATE MEDICAL REPORTS
# =========================
if __name__ == "__main__":
    
    print("\n" + "="*70)
    print("VLM MEDICAL REPORT GENERATION SYSTEM")
    print("="*70)
    print("Multimodal architecture: Vision Feature Encoder → LLM Report Generator")
    print("="*70)
    
    # Generate reports for multiple images
    num_reports = min(3, len(vision_features))
    
    for idx in range(num_reports):
        print(f"\n[Report {idx+1}] Analyzing scan...")
        try:
            report = generate_text(vision_features[idx:idx+1], max_len=100)
            print(f"\n{'─'*70}")
            print(f"CLINICAL IMPRESSION - Image {idx+1}:")
            print(f"{'─'*70}")
            print(report)
            print(f"{'─'*70}\n")
        except Exception as e:
            print(f"Error generating report: {e}")
            import traceback
            traceback.print_exc()
    
    print("="*70)
    print("Report generation complete.")
    print("="*70)



VLM MEDICAL REPORT GENERATION SYSTEM
Multimodal architecture: Vision Feature Encoder → LLM Report Generator

[Report 1] Analyzing scan...

──────────────────────────────────────────────────────────────────────
CLINICAL IMPRESSION - Image 1:
──────────────────────────────────────────────────────────────────────
, who was a member of the House of Lords, was a member of the House of Lords, and was a member of the House of Lords.

The House of Lords was a body of members of the House of Lords, and was a body of members of the House of Lords.

The House of Lords was a body of members of the House of Lords, and was a body of members of the House of Lords.

The House of Lords was a body of members of
──────────────────────────────────────────────────────────────────────


[Report 2] Analyzing scan...

──────────────────────────────────────────────────────────────────────
CLINICAL IMPRESSION - Image 2:
──────────────────────────────────────────────────────────────────────
, who was a member o

Testing simplified VLM generation...
Vision features shape: torch.Size([1, 768])
Clinical context shape: torch.Size([1, 512])
Vision embedding shape: torch.Size([1, 512])
Context repeated shape: torch.Size([1, 512])
Fused embedding shape: torch.Size([1, 512])
LLM input shape: torch.Size([1, 768])
LLM input expanded shape: torch.Size([1, 1, 768])
LLM output logits shape: torch.Size([1, 1, 50257])
Next token logits shape: torch.Size([1, 50257])
Next token: tensor([11]) (value: 11)
Decoded token: ','

Test successful! Ready to generate reports.
